# Binance daily feature and rolling-selection analysis

This notebook applies the approved Silva-derived feature definitions to the immutable Binance Spot daily klines. It calculates 22 features independently for each cryptocurrency and then refits the approved variance and correlation filters inside rolling windows of 60, 75, and 90 daily observations.

The canonical `csv.gz` files are used because the `.xlsx` files are review copies of the same collected observations. Raw files are never modified. Derived feature tables are written under `data/processed/`, while selection histories, summaries, metadata, and the figure are written under `outputs/`.

This historical exploratory report retains its full-span, gap-rejecting behavior. The pilot/development pipeline with approved internal-gap filling is exposed by `python -m tcc_crypto.daily`; notebooks 14 and 15 show those stages. Paths and metadata helpers are shared. Install the package in editable mode before running this notebook.


## Methodological boundaries

For every cryptocurrency and prediction date, feature selection is fitted only on the immediately preceding training window. The next observation is transformed with those fitted selectors but never participates in fitting. Indicator warm-up rows are removed only from the leading edge. A missing day or an internal missing feature blocks that cryptocurrency instead of triggering imputation.

Parabolic SAR currently reproduces the provisional zero-valued parameters from the Silva reference and remains marked for later verification. This notebook does not construct targets, scale features, train an SVM, or interpret selection frequency as predictive performance.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from importlib.metadata import version
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


from tcc_crypto.paths import (
    repository_root, LEGACY_FEATURE_TABLES_DIR, FEATURE_ANALYSIS_FILES,
    FEATURE_WORKBOOK_PACKETS_DIR, FEATURE_WORKBOOK_QA_DIR, FEATURE_WORKBOOK_BUILDER,
    LEGACY_FEATURE_TABLE_TEMPLATE,
)
from tcc_crypto.data.loaders import find_symbol_csv
from tcc_crypto.reporting.provenance import execution_metadata

REPOSITORY_ROOT = repository_root()

from tcc_crypto.binance_kline_collection import load_collection_config
from tcc_crypto.daily_feature_selection import (
    remove_leading_feature_warmup,
    summarize_rolling_feature_selection,
)
from tcc_crypto.daily_features import (
    ACTIVE_FEATURES,
    calculate_daily_features,
    load_daily_feature_config,
    load_daily_feature_input,
)

## Load the approved configuration

By default, the notebook processes the complete configured universe. For a development smoke test only, set `TCC_CRYPTO_FEATURE_ANALYSIS_SYMBOL_LIMIT` before starting the kernel. Any limited execution is stored in a separate scope directory and identified in the metadata.

In [ ]:
FEATURE_CONFIG_PATH = REPOSITORY_ROOT / 'configs/binance_daily_features.json'
feature_config = load_daily_feature_config(FEATURE_CONFIG_PATH)
collection_config = load_collection_config(
    REPOSITORY_ROOT / feature_config.collection_config
)

if collection_config.interval != feature_config.interval:
    raise ValueError('Collection and feature intervals do not match.')

symbol_limit_text = os.environ.get('TCC_CRYPTO_FEATURE_ANALYSIS_SYMBOL_LIMIT')
if symbol_limit_text is None:
    symbols = collection_config.symbols
    run_scope = 'full'
else:
    symbol_limit = int(symbol_limit_text)
    if symbol_limit < 1:
        raise ValueError('The smoke-test symbol limit must be positive.')
    symbols = collection_config.symbols[:symbol_limit]
    run_scope = f'smoke_{symbol_limit}_symbols'

raw_directory = REPOSITORY_ROOT / collection_config.output_directory
feature_output_directory = REPOSITORY_ROOT / LEGACY_FEATURE_TABLES_DIR
analysis_output_directory = (
    REPOSITORY_ROOT / feature_config.analysis_output_directory / run_scope
)
feature_output_directory.mkdir(parents=True, exist_ok=True)
analysis_output_directory.mkdir(parents=True, exist_ok=True)

configuration_summary = pd.DataFrame(
    [
        ('Feature set', feature_config.feature_set_id),
        ('Collection', collection_config.collection_id),
        ('Interval', feature_config.interval),
        ('Timezone', feature_config.time_zone),
        ('Volume input', feature_config.volume_field),
        ('Active features', len(feature_config.active_features)),
        ('Training windows', feature_config.feature_selection.training_window_sizes),
        ('Variance threshold', feature_config.feature_selection.variance_threshold),
        ('Correlation threshold', feature_config.feature_selection.correlation_threshold),
        ('Correlation method', feature_config.feature_selection.correlation_method),
        ('Run scope', run_scope),
        ('Requested symbols', len(symbols)),
    ],
    columns=['Setting', 'Value'],
)
display(configuration_summary)

## Calculate and validate daily features

Each symbol is processed independently. A successful table contains one UTC-indexed row per complete day and the 22 approved features. Blocked symbols remain in the manifest with their diagnostic; the notebook does not silently remove or repair their raw observations.

In [ ]:
feature_tables: dict[str, pd.DataFrame] = {}
manifest_rows: list[dict[str, object]] = []
raw_sources: dict[str, str] = {}
missing_dates_by_symbol: dict[str, list[pd.Timestamp]] = {}

for symbol in symbols:
    candidates = sorted((raw_directory / symbol).glob(f'{symbol}_1d_*.csv.gz'))
    try:
        source_path = find_symbol_csv(raw_directory, symbol)
        raw_sources[symbol] = str(candidates[0].relative_to(REPOSITORY_ROOT))
        raw_values = load_daily_feature_input(source_path)
        calculated = calculate_daily_features(raw_values)
        complete = remove_leading_feature_warmup(calculated)
        output_path = feature_output_directory / LEGACY_FEATURE_TABLE_TEMPLATE.format(symbol=symbol)
        complete.to_csv(output_path, compression='gzip', index_label='open_time_utc')
        feature_tables[symbol] = complete
        missing_dates_by_symbol[symbol] = []
        manifest_rows.append(
            {
                'symbol': symbol,
                'status': 'READY',
                'raw_rows': len(raw_values),
                'warmup_rows_removed': len(calculated) - len(complete),
                'complete_feature_rows': len(complete),
                'first_feature_time_utc': complete.index.min(),
                'last_feature_time_utc': complete.index.max(),
                'feature_file': str(output_path.relative_to(REPOSITORY_ROOT)),
                'missing_utc_days': 0,
                'diagnostic': '',
            }
        )
    except Exception as error:
        missing_dates: list[pd.Timestamp] = []
        if len(candidates) == 1:
            raw_sources[symbol] = str(candidates[0].relative_to(REPOSITORY_ROOT))
            timestamp_values = pd.read_csv(candidates[0], usecols=['open_time_ms'])
            observed_dates = pd.DatetimeIndex(
                pd.to_datetime(timestamp_values['open_time_ms'], unit='ms', utc=True)
            )
            expected_dates = pd.date_range(
                observed_dates.min(), observed_dates.max(), freq='D', tz='UTC'
            )
            missing_dates = expected_dates.difference(observed_dates).tolist()
        missing_dates_by_symbol[symbol] = missing_dates
        manifest_rows.append(
            {
                'symbol': symbol,
                'status': 'BLOCKED',
                'raw_rows': pd.NA,
                'warmup_rows_removed': pd.NA,
                'complete_feature_rows': pd.NA,
                'first_feature_time_utc': pd.NaT,
                'last_feature_time_utc': pd.NaT,
                'feature_file': '',
                'missing_utc_days': len(missing_dates),
                'diagnostic': f'{type(error).__name__}: {error}',
            }
        )

feature_manifest = pd.DataFrame(manifest_rows)
manifest_path = analysis_output_directory / FEATURE_ANALYSIS_FILES['manifest']
feature_manifest.to_csv(manifest_path, index=False)
display(feature_manifest)
display(
    feature_manifest['status']
    .value_counts(dropna=False)
    .rename_axis('Status')
    .reset_index(name='Symbols')
)

## Inspect one feature table

The preview confirms the row-level structure: one cryptocurrency, one UTC day per row, and 22 feature columns. It is descriptive only and does not select a preferred asset.

In [ ]:
if not feature_tables:
    raise RuntimeError('No symbol produced a valid feature table.')
example_symbol = 'BTCUSDT' if 'BTCUSDT' in feature_tables else next(iter(feature_tables))
example_features = feature_tables[example_symbol]
print(f'Example symbol: {example_symbol}')
display(example_features.head())
display(example_features.tail())
display(example_features.describe().T)

## Run rolling feature selection

This is the computationally intensive stage. For each ready symbol and each window size, both filters are refitted for every next-day observation. The history records dates and selected feature names, while compact tables report selection frequency and the number of retained features. Selection frequency describes the filters' behavior; it is not model importance.

In [ ]:
history_tables: list[pd.DataFrame] = []
frequency_tables: list[pd.DataFrame] = []
window_tables: list[pd.DataFrame] = []

for position, (symbol, features) in enumerate(feature_tables.items(), start=1):
    print(f'[{position}/{len(feature_tables)}] Selecting features for {symbol}')
    summary = summarize_rolling_feature_selection(
        features,
        feature_config.feature_selection,
    )
    history_tables.append(summary.history.assign(symbol=symbol))
    frequency_tables.append(summary.feature_frequency.assign(symbol=symbol))
    window_tables.append(summary.window_summary.assign(symbol=symbol))

selection_history = pd.concat(history_tables, ignore_index=True)
selection_frequency = pd.concat(frequency_tables, ignore_index=True)
window_summary = pd.concat(window_tables, ignore_index=True)

history_path = analysis_output_directory / FEATURE_ANALYSIS_FILES['history']
frequency_path = analysis_output_directory / FEATURE_ANALYSIS_FILES['frequency']
window_summary_path = analysis_output_directory / FEATURE_ANALYSIS_FILES['windows']
selection_history.to_csv(history_path, index=False, compression='gzip')
selection_frequency.to_csv(frequency_path, index=False)
window_summary.to_csv(window_summary_path, index=False)

display(window_summary)
display(
    selection_frequency.sort_values(
        ['window_size', 'selection_rate'],
        ascending=[True, False],
    ).head(30)
)

## Selection-frequency overview

The heatmap averages each feature's selection rate across ready cryptocurrencies. A value of 100% means that the feature survived both filters in every rolling fit for that window size; it does not imply that the feature predicts the target.

In [ ]:
frequency_overview = (
    selection_frequency.groupby(['feature', 'window_size'])['selection_rate']
    .mean()
    .unstack('window_size')
    .reindex(ACTIVE_FEATURES)
)
display(frequency_overview.style.format('{:.1%}'))

figure_height = max(7.0, 0.35 * len(frequency_overview))
fig, ax = plt.subplots(figsize=(8, figure_height))
image = ax.imshow(frequency_overview.to_numpy(), aspect='auto', vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(frequency_overview.columns)), frequency_overview.columns)
ax.set_yticks(range(len(frequency_overview.index)), frequency_overview.index)
ax.set_xlabel('Rolling training window (daily observations)')
ax.set_ylabel('Feature')
ax.set_title('Mean feature-selection rate across ready Binance USDT pairs')
colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label('Mean selection rate')
fig.tight_layout()
figure_path = analysis_output_directory / FEATURE_ANALYSIS_FILES['figure']
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()

## Generate one Excel workbook per cryptocurrency

Every ready cryptocurrency receives a workbook with `Summary`, `Features`, `Selection frequency`, `Window summary`, and `Selection history` worksheets. A blocked cryptocurrency still receives a diagnostic workbook, but no feature values are fabricated. Workbook generation uses the bundled artifact tool and requires `TCC_CRYPTO_NODE` and `TCC_CRYPTO_NODE_MODULES` in the notebook environment.

In [ ]:
packet_directory = REPOSITORY_ROOT / FEATURE_WORKBOOK_PACKETS_DIR / run_scope
packet_directory.mkdir(parents=True, exist_ok=True)
for old_packet in packet_directory.glob('*.json'):
    old_packet.unlink()

manifest_by_symbol = feature_manifest.set_index('symbol')
for symbol in symbols:
    manifest_record = manifest_by_symbol.loc[symbol]
    packet: dict[str, object] = {
        'symbol': symbol,
        'status': manifest_record['status'],
        'diagnostic': manifest_record['diagnostic'],
        'source': raw_sources.get(symbol, ''),
        'missing_dates': [
            date.isoformat() for date in missing_dates_by_symbol.get(symbol, [])
        ],
        'parameters': {
            'active_features': list(feature_config.active_features),
            'variance_threshold': feature_config.feature_selection.variance_threshold,
            'correlation_threshold': feature_config.feature_selection.correlation_threshold,
            'training_window_sizes': list(
                feature_config.feature_selection.training_window_sizes
            ),
        },
    }
    if symbol in feature_tables:
        feature_records = json.loads(
            feature_tables[symbol]
            .rename_axis('open_time_utc')
            .reset_index()
            .to_json(orient='records', date_format='iso')
        )
        history_records = json.loads(
            selection_history.loc[selection_history['symbol'] == symbol]
            .drop(columns='symbol')
            .to_json(orient='records', date_format='iso')
        )
        frequency_records = json.loads(
            selection_frequency.loc[selection_frequency['symbol'] == symbol]
            .drop(columns='symbol')
            .to_json(orient='records')
        )
        window_records = json.loads(
            window_summary.loc[window_summary['symbol'] == symbol]
            .drop(columns='symbol')
            .to_json(orient='records')
        )
        packet.update(
            {
                'coverage': {
                    'raw_rows': int(manifest_record['raw_rows']),
                    'warmup_rows_removed': int(manifest_record['warmup_rows_removed']),
                    'complete_feature_rows': int(manifest_record['complete_feature_rows']),
                    'first_feature_time_utc': manifest_record['first_feature_time_utc'].isoformat(),
                    'last_feature_time_utc': manifest_record['last_feature_time_utc'].isoformat(),
                },
                'features': feature_records,
                'selection_history': history_records,
                'selection_frequency': frequency_records,
                'window_summary': window_records,
            }
        )
    packet_path = packet_directory / f'{symbol}.json'
    packet_path.write_text(
        json.dumps(packet, ensure_ascii=False),
        encoding='utf-8',
    )

print(f'Workbook packets: {len(list(packet_directory.glob("*.json")))}')

In [ ]:
node_executable = os.environ.get('TCC_CRYPTO_NODE')
node_modules = os.environ.get('TCC_CRYPTO_NODE_MODULES')
if not node_executable or not node_modules:
    raise RuntimeError(
        'Set TCC_CRYPTO_NODE and TCC_CRYPTO_NODE_MODULES before workbook generation.'
    )

builder_source = REPOSITORY_ROOT / FEATURE_WORKBOOK_BUILDER
workbook_output_directory = analysis_output_directory / 'workbooks'
workbook_qa_directory = (
    REPOSITORY_ROOT / FEATURE_WORKBOOK_QA_DIR / run_scope
)
with tempfile.TemporaryDirectory(prefix='tcc_crypto_feature_workbooks_') as temporary_directory:
    temporary_path = Path(temporary_directory)
    builder_path = temporary_path / builder_source.name
    shutil.copy2(builder_source, builder_path)
    (temporary_path / 'node_modules').symlink_to(
        Path(node_modules), target_is_directory=True
    )
    subprocess.run(
        [
            node_executable,
            str(builder_path),
            str(packet_directory),
            str(workbook_output_directory),
            str(workbook_qa_directory),
            str(len(symbols)),
        ],
        cwd=REPOSITORY_ROOT,
        check=True,
    )

workbook_manifest_path = workbook_output_directory / FEATURE_ANALYSIS_FILES['workbook_manifest']
workbook_manifest = pd.read_csv(workbook_manifest_path)
if len(workbook_manifest) != len(symbols):
    raise RuntimeError('Workbook manifest does not cover every requested symbol.')
display(workbook_manifest)

## Record reproducibility metadata

The final cell records the execution timestamp, code revision, package versions, requested and processed universes, configuration, and output locations. A dirty revision is recorded explicitly because uncommitted research code cannot be reproduced from the commit alone.

In [ ]:
provenance = execution_metadata(REPOSITORY_ROOT)
git_revision = provenance['code_revision']
git_dirty = provenance['working_tree_dirty']
metadata = {
    'analysis_id': f'{feature_config.feature_set_id}_{run_scope}',
    'execution_time_utc': datetime.now(timezone.utc).isoformat(),
    'code_revision': git_revision,
    'working_tree_dirty': git_dirty,
    'feature_config': str(FEATURE_CONFIG_PATH.relative_to(REPOSITORY_ROOT)),
    'collection_config': str(feature_config.collection_config),
    'run_scope': run_scope,
    'requested_symbols': list(symbols),
    'ready_symbols': list(feature_tables),
    'blocked_symbols': feature_manifest.loc[
        feature_manifest['status'] == 'BLOCKED', 'symbol'
    ].tolist(),
    'active_features': list(feature_config.active_features),
    'features_requiring_verification': list(
        feature_config.features_requiring_verification
    ),
    'selection': {
        'variance_threshold': feature_config.feature_selection.variance_threshold,
        'correlation_threshold': feature_config.feature_selection.correlation_threshold,
        'correlation_method': feature_config.feature_selection.correlation_method,
        'training_window_sizes': list(
            feature_config.feature_selection.training_window_sizes
        ),
        'window_type': feature_config.feature_selection.window_type,
        'refit_frequency': feature_config.feature_selection.refit_frequency,
        'fit_scope': feature_config.feature_selection.fit_scope,
    },
    'package_versions': {
        package: version(package)
        for package in ['feature-engine', 'matplotlib', 'pandas', 'scikit-learn', 'TA-Lib']
    },
    'artifacts': {
        'feature_directory': str(feature_output_directory.relative_to(REPOSITORY_ROOT)),
        'analysis_directory': str(analysis_output_directory.relative_to(REPOSITORY_ROOT)),
        'feature_manifest': str(manifest_path.relative_to(REPOSITORY_ROOT)),
        'selection_history': str(history_path.relative_to(REPOSITORY_ROOT)),
        'selection_frequency': str(frequency_path.relative_to(REPOSITORY_ROOT)),
        'window_summary': str(window_summary_path.relative_to(REPOSITORY_ROOT)),
        'selection_figure': str(figure_path.relative_to(REPOSITORY_ROOT)),
        'workbook_directory': str(workbook_output_directory.relative_to(REPOSITORY_ROOT)),
        'workbook_manifest': str(workbook_manifest_path.relative_to(REPOSITORY_ROOT)),
    },
}
metadata_path = analysis_output_directory / FEATURE_ANALYSIS_FILES['metadata']
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print(f'Metadata: {metadata_path}')
print(f'Ready symbols: {len(feature_tables)} of {len(symbols)}')
print(f'Rolling selection rows: {len(selection_history):,}')

## Interpretation limits and next stage

The resulting tables verify feature availability and the stability of the approved filters across training-window lengths. They do not show whether a selected feature improves classification. That question requires approved targets, temporal train/validation/test boundaries, training-only scaling, and an SVM evaluation protocol.

Any blocked symbol must remain outside downstream modeling until its diagnostic receives an explicit methodology decision. In the current raw collection, `LUNAUSDT` is expected to be blocked because its observed span contains missing UTC days.